# Smart Governance Analytics - Exploratory Data Analysis and Model Development

This notebook provides a comprehensive analysis of governance data for East African regions and demonstrates the development of machine learning models for resource allocation prediction.

## Table of Contents
1. [Data Loading and Overview](#data-loading)
2. [Exploratory Data Analysis](#eda)
3. [Data Preprocessing](#preprocessing)
4. [Feature Engineering](#feature-engineering)
5. [Model Development](#model-development)
6. [Model Evaluation](#model-evaluation)
7. [Insights and Recommendations](#insights)
8. [Future Extensions](#extensions)

## 1. Data Loading and Overview {#data-loading}

Let's start by importing necessary libraries and loading our governance dataset.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Import custom modules
import sys
sys.path.append('../src')
from config import SAMPLE_DATA_PATH
from etl import DataETL
from model import ResourceAllocationModel

In [ ]:
# Load the data
data = pd.read_csv(SAMPLE_DATA_PATH)
print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")
data.head()

In [ ]:
# Basic information about the dataset
print("Dataset Info:")
data.info()
print("\n" + "="*50)
print("Missing Values:")
print(data.isnull().sum())
print("\n" + "="*50)
print("Summary Statistics:")
data.describe()

## 2. Exploratory Data Analysis {#eda}

Let's explore the data to understand patterns, relationships, and distributions.

In [ ]:
# Distribution of regions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Region distribution
data['region'].value_counts().plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Districts by Region')
axes[0,0].set_xlabel('Region')
axes[0,0].set_ylabel('Number of Districts')

# Population distribution
data['population'].hist(bins=15, ax=axes[0,1], color='lightgreen')
axes[0,1].set_title('Population Distribution')
axes[0,1].set_xlabel('Population')
axes[0,1].set_ylabel('Frequency')

# Income distribution
data['income_per_capita'].hist(bins=15, ax=axes[1,0], color='coral')
axes[1,0].set_title('Income per Capita Distribution')
axes[1,0].set_xlabel('Income per Capita ($)')
axes[1,0].set_ylabel('Frequency')

# Current allocation distribution
data['current_allocation'].hist(bins=15, ax=axes[1,1], color='gold')
axes[1,1].set_title('Current Allocation Distribution')
axes[1,1].set_xlabel('Current Allocation ($)')
axes[1,1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Regional comparison using box plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Population by region
sns.boxplot(data=data, x='region', y='population', ax=axes[0,0])
axes[0,0].set_title('Population by Region')
axes[0,0].tick_params(axis='x', rotation=45)

# Income by region
sns.boxplot(data=data, x='region', y='income_per_capita', ax=axes[0,1])
axes[0,1].set_title('Income per Capita by Region')
axes[0,1].tick_params(axis='x', rotation=45)

# Health index by region
sns.boxplot(data=data, x='region', y='health_index', ax=axes[1,0])
axes[1,0].set_title('Health Index by Region')
axes[1,0].tick_params(axis='x', rotation=45)

# Education index by region
sns.boxplot(data=data, x='region', y='education_index', ax=axes[1,1])
axes[1,1].set_title('Education Index by Region')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
numeric_cols = ['population', 'income_per_capita', 'health_index', 'education_index', 'current_allocation']
correlation_matrix = data[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix of Key Variables')
plt.tight_layout()
plt.show()

print("\nStrongest correlations with current allocation:")
corr_with_allocation = correlation_matrix['current_allocation'].abs().sort_values(ascending=False)
print(corr_with_allocation[1:])  # Exclude self-correlation

In [ ]:
# Scatter plot matrix for key relationships
fig = px.scatter_matrix(
    data,
    dimensions=['population', 'income_per_capita', 'health_index', 'education_index', 'current_allocation'],
    color='region',
    title='Scatter Plot Matrix of Key Variables by Region',
    height=800
)
fig.show()

## 3. Data Preprocessing {#preprocessing}

Now let's preprocess the data using our ETL pipeline.

In [ ]:
# Initialize ETL pipeline
etl = DataETL()

# Load and validate data
raw_data = etl.extract()
validated_data = etl.validate_data(raw_data)

print(f"Original data shape: {raw_data.shape}")
print(f"Validated data shape: {validated_data.shape}")

In [ ]:
# Transform the data
transformed_data = etl.transform(validated_data)

print(f"Transformed data shape: {transformed_data.shape}")
print(f"\nNew columns created:")
new_columns = set(transformed_data.columns) - set(raw_data.columns)
for col in new_columns:
    print(f"  - {col}")

transformed_data.head()

In [ ]:
# Prepare features for modeling
X, y = etl.prepare_features(transformed_data)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns:")
for i, col in enumerate(X.columns):
    print(f"  {i+1}. {col}")

## 4. Feature Engineering {#feature-engineering}

Let's explore the engineered features and their relationships.

In [ ]:
# Analyze engineered features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Population density
transformed_data['population_density'].hist(bins=15, ax=axes[0,0], color='lightblue')
axes[0,0].set_title('Population Density Distribution')
axes[0,0].set_xlabel('Population Density')

# Socioeconomic index
transformed_data['socioeconomic_index'].hist(bins=15, ax=axes[0,1], color='lightcoral')
axes[0,1].set_title('Socioeconomic Index Distribution')
axes[0,1].set_xlabel('Socioeconomic Index')

# Income category distribution
transformed_data['income_category'].value_counts().plot(kind='bar', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Income Category Distribution')
axes[1,0].set_xlabel('Income Category')
axes[1,0].tick_params(axis='x', rotation=45)

# Socioeconomic index vs current allocation
axes[1,1].scatter(transformed_data['socioeconomic_index'], transformed_data['current_allocation'], alpha=0.7)
axes[1,1].set_title('Socioeconomic Index vs Current Allocation')
axes[1,1].set_xlabel('Socioeconomic Index')
axes[1,1].set_ylabel('Current Allocation')

plt.tight_layout()
plt.show()

## 5. Model Development {#model-development}

Let's train and compare different models for resource allocation prediction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Train Random Forest model
rf_model = ResourceAllocationModel(model_type="random_forest")
rf_train_metrics = rf_model.train(X_train, y_train)
rf_test_metrics = rf_model.evaluate(X_test, y_test)

print("Random Forest Results:")
print(f"Training R²: {rf_train_metrics['train_r2']:.4f}")
print(f"Test R²: {rf_test_metrics['test_r2']:.4f}")
print(f"Test RMSE: {rf_test_metrics['test_rmse']:.2f}")

In [ ]:
# Train Linear Regression model for comparison
lr_model = ResourceAllocationModel(model_type="linear_regression")
lr_train_metrics = lr_model.train(X_train, y_train)
lr_test_metrics = lr_model.evaluate(X_test, y_test)

print("Linear Regression Results:")
print(f"Training R²: {lr_train_metrics['train_r2']:.4f}")
print(f"Test R²: {lr_test_metrics['test_r2']:.4f}")
print(f"Test RMSE: {lr_test_metrics['test_rmse']:.2f}")

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'Linear Regression'],
    'Train R²': [rf_train_metrics['train_r2'], lr_train_metrics['train_r2']],
    'Test R²': [rf_test_metrics['test_r2'], lr_test_metrics['test_r2']],
    'Test RMSE': [rf_test_metrics['test_rmse'], lr_test_metrics['test_rmse']],
    'Test MAE': [rf_test_metrics['test_mae'], lr_test_metrics['test_mae']]
})

print("Model Comparison:")
print(comparison_df.round(4))

## 6. Model Evaluation {#model-evaluation}

Let's evaluate our best model in detail.

In [ ]:
# Use the Random Forest model for detailed analysis
best_model = rf_model

# Feature importance
if best_model.feature_importance is not None:
    plt.figure(figsize=(10, 6))
    top_features = best_model.feature_importance.head(10)
    
    sns.barplot(data=top_features, x='importance', y='feature', palette='viridis')
    plt.title('Top 10 Feature Importances')
    plt.xlabel('Importance')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()
    
    print("\nTop 5 Most Important Features:")
    for idx, row in top_features.head().iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")

In [ ]:
# Predictions vs Actual values
y_pred = best_model.predict(X_test)

plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Predictions vs Actual Values')

# Add R² score to plot
r2 = r2_score(y_test, y_pred)
plt.text(0.05, 0.95, f'R² = {r2:.4f}', transform=plt.gca().transAxes, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Residuals analysis
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.7)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted Values')

# Residuals distribution
axes[1].hist(residuals, bins=15, alpha=0.7, color='skyblue')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f}")
print(f"Standard deviation of residuals: {residuals.std():.2f}")

## 7. Insights and Recommendations {#insights}

Let's extract actionable insights from our analysis.

In [ ]:
# Regional analysis for policy recommendations
regional_analysis = transformed_data.groupby('region').agg({
    'population': ['sum', 'mean'],
    'income_per_capita': 'mean',
    'health_index': 'mean',
    'education_index': 'mean',
    'socioeconomic_index': 'mean',
    'current_allocation': ['sum', 'mean']
}).round(2)

print("Regional Analysis Summary:")
print(regional_analysis)

In [ ]:
# Identify districts that may need additional resources
# Calculate efficiency: allocation per capita vs socioeconomic need
efficiency_analysis = transformed_data.copy()
efficiency_analysis['allocation_per_capita'] = efficiency_analysis['current_allocation'] / efficiency_analysis['population']
efficiency_analysis['need_index'] = 1 - efficiency_analysis['socioeconomic_index']  # Higher need = lower socioeconomic index
efficiency_analysis['efficiency_ratio'] = efficiency_analysis['allocation_per_capita'] / efficiency_analysis['need_index']

# Districts with low efficiency (high need, low allocation per capita)
low_efficiency = efficiency_analysis.nsmallest(5, 'efficiency_ratio')[['district', 'region', 'allocation_per_capita', 'need_index', 'efficiency_ratio']]

print("Districts that may need additional resource allocation:")
print(low_efficiency.round(2))

In [ ]:
# Example predictions for policy scenarios
print("Policy Scenario Analysis:")
print("\n1. High-need district with improved education:")

# Scenario: Improving education in a low-performing district
scenario_prediction = best_model.predict_single(
    region="North",
    district="Moroto",
    population=140000,
    income_per_capita=280,
    health_index=0.42,
    education_index=0.65  # Improved from 0.45
)

print(f"   Predicted allocation with education improvement: ${scenario_prediction:,.0f}")

# Compare with current
current_moroto = data[data['district'] == 'Moroto']['current_allocation'].iloc[0]
print(f"   Current allocation: ${current_moroto:,.0f}")
print(f"   Difference: ${scenario_prediction - current_moroto:,.0f} ({((scenario_prediction - current_moroto) / current_moroto * 100):+.1f}%)")

## 8. Future Extensions {#extensions}

This analysis framework can be extended in several ways:

### 8.1 Additional Models
- **Fraud Detection**: Use anomaly detection to identify unusual spending patterns
- **Sentiment Analysis**: Analyze public opinion from social media and surveys
- **Time Series Forecasting**: Predict future resource needs
- **Classification Models**: Categorize districts by development level

### 8.2 New Data Sources
- **APIs**: Real-time data from government databases
- **Social Media**: Public sentiment and feedback
- **Satellite Data**: Infrastructure and development indicators
- **Economic Indicators**: Market data and trade statistics

### 8.3 Advanced Analytics
- **Causal Inference**: Understand cause-effect relationships
- **Optimization**: Optimal resource allocation algorithms
- **Multi-objective Analysis**: Balance multiple policy goals
- **Simulation**: Model policy scenario outcomes

In [ ]:
# Example: Setting up for time series analysis (stub)
print("Future Extension Example: Time Series Setup")
print("\nTo implement time series forecasting:")
print("1. Collect historical data with multiple years")
print("2. Add temporal features (month, quarter, year)")
print("3. Implement ARIMA, Prophet, or LSTM models")
print("4. Forecast future resource needs")

# Placeholder for future implementation
# from statsmodels.tsa.arima.model import ARIMA
# from prophet import Prophet

print("\nExample data structure for time series:")
time_series_example = pd.DataFrame({
    'date': pd.date_range('2020-01-01', '2023-12-31', freq='M'),
    'district': 'Kampala',
    'allocation': np.random.normal(25000000, 2000000, 48)
})
print(time_series_example.head())

## Conclusion

This notebook demonstrates a complete machine learning pipeline for smart governance analytics:

1. **Data Analysis**: Comprehensive EDA revealing regional disparities
2. **Feature Engineering**: Created meaningful indicators like socioeconomic index
3. **Model Development**: Trained predictive models with good performance
4. **Policy Insights**: Identified districts needing additional resources
5. **Extensibility**: Framework ready for additional models and data sources

The solution provides a solid foundation for evidence-based policy making in East African governance.